# RAVDESS Evaluation: Speech Emotion Metrics + Multimodal Fusion Ablation

Closes reviewer points A and B. Runs on your existing working env (torch, ultralytics, transformers, opencv, scipy, sklearn, matplotlib). **No new packages.**

**Part A** evaluates the HuBERT speech branch on RAVDESS audio (accuracy, macro-F1, confusion matrix).
**Part B** runs the late-fusion ablation: sweeps alpha, compares fused vs vision-only vs speech-only, and tests the confidence-weighted dynamic-alpha rule the paper proposes.

Scope: the SUPERB HuBERT model predicts only 4 emotions (angry, happy, neutral, sad), so the multimodal evaluation is on those 4 classes. RAVDESS 'calm' is folded into 'neutral' (standard practice); fearful/disgust/surprised are excluded because the speech model cannot output them.

### Download first (datasets only)
From the RAVDESS Zenodo page (record 1188976):
- `Audio_Speech_Actors_01-24.zip` -> required for Part A
- `Video_Speech_Actors_01-24.zip` -> required for Part B

Unzip each so you have `Actor_01 ... Actor_24` folders, then set the two paths in the config cell below.


## Config and imports

In [ ]:
import os
os.environ["USE_TF"] = "0"

import glob, math, csv
import numpy as np
import cv2, torch
from collections import defaultdict
import scipy.io.wavfile as wavfile
from scipy.signal import resample_poly
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
import matplotlib.pyplot as plt

# >>> EDIT THESE PATHS <<<
RAVDESS_AUDIO_DIR = r"data/ravdess\Audio_Speech_Actors_01-24"
RAVDESS_VIDEO_DIR = r"data/ravdess\Video_Speech_Actors_01-24"
DET_WEIGHTS       = r"runs/detect/face_detector\weights\best.pt"
FER_WEIGHTS       = r"runs/classify/fer_yolo_cls\weights\best.pt"

OUT_DIR = os.path.join(os.getcwd(), "eval_outputs"); os.makedirs(OUT_DIR, exist_ok=True)

# runtime caps (raise for fuller runs)
AUDIO_MAX_PER_CLASS  = None   # None = use all audio clips for Part A
FUSION_MAX_PER_CLASS = 120    # clips per class for Part B (video reading is the slow part)
FUSION_FRAMES_PER_CLIP = 6    # frames sampled per video clip
PAD = 0.30                    # crop padding, matches the paper

COMMON4 = ["angry", "happy", "neutral", "sad"]   # fixed class order everywhere
# RAVDESS emotion code (3rd filename field) -> COMMON4 ; calm(02)->neutral ; 06/07/08 excluded
RAV_EMO = {"01":"neutral","02":"neutral","03":"happy","04":"sad","05":"angry"}
DEV = 0 if torch.cuda.is_available() else -1
print("output dir:", OUT_DIR, "| device:", "GPU" if DEV==0 else "CPU")


## Load models (YOLO detector + FER classifier + HuBERT)

In [ ]:
from ultralytics import YOLO
from transformers import pipeline

det_model = YOLO(DET_WEIGHTS)
emo_model = YOLO(FER_WEIGHTS)
FER_NAMES = [emo_model.names[i] for i in range(len(emo_model.names))]
print("FER class order:", FER_NAMES)
FER_IDX4 = [FER_NAMES.index(c) for c in COMMON4]   # map FER 7-class -> COMMON4

ser = pipeline("audio-classification", model="superb/hubert-large-superb-er", device=DEV)
print("HuBERT SER labels:", ser.model.config.id2label)
print("\u2705 models + SER pipeline ready")


## Helpers

In [ ]:
def parse_emotion(path):
    """RAVDESS filename: modality-voc-emotion-intensity-statement-rep-actor.ext"""
    base = os.path.basename(path).split(".")[0]
    parts = base.split("-")
    if len(parts) != 7: return None, None
    return parts[2], "-".join(parts[1:])   # (emotion_code, key-without-modality)

def load_wav_16k(path):
    sr, x = wavfile.read(path)
    x = x.astype(np.float32)
    if x.ndim > 1: x = x.mean(axis=1)            # stereo -> mono
    if sr != 16000:
        g = math.gcd(sr, 16000)
        x = resample_poly(x, 16000 // g, sr // g)
    m = np.max(np.abs(x)) + 1e-9
    return (x / m).astype(np.float32)

def ser_label_to_common(lbl):
    l = lbl.lower()
    if l.startswith("neu"): return "neutral"
    if l.startswith("hap"): return "happy"
    if l.startswith("ang"): return "angry"
    if l.startswith("sad"): return "sad"
    return None

def speech_probs4(path):
    """Return 4-dim prob vector over COMMON4 from HuBERT."""
    x = load_wav_16k(path)
    out = ser({"raw": x, "sampling_rate": 16000}, top_k=None)
    v = np.zeros(4, dtype=np.float32)
    for d in out:
        c = ser_label_to_common(d["label"])
        if c is not None: v[COMMON4.index(c)] = d["score"]
    s = v.sum()
    return v / s if s > 0 else v

def biggest_box(det):
    if det.boxes is None or len(det.boxes) == 0: return None
    b = det.boxes.xyxy.cpu().numpy()
    a = (b[:,2]-b[:,0])*(b[:,3]-b[:,1])
    return b[int(np.argmax(a))]

def face_probs4_from_video(path, n_frames=FUSION_FRAMES_PER_CLIP):
    """Sample frames, run detector+FER, return averaged 4-dim prob vector (or None)."""
    cap = cv2.VideoCapture(path)
    if not cap.isOpened(): return None
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT)) or 1
    idxs = np.linspace(int(total*0.2), int(total*0.8), n_frames).astype(int)
    acc, used = np.zeros(4, dtype=np.float32), 0
    for fi in idxs:
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(fi))
        ok, frame = cap.read()
        if not ok: continue
        det = det_model.predict(frame, conf=0.35, iou=0.5, device=DEV if DEV==0 else "cpu", verbose=False)[0]
        box = biggest_box(det)
        if box is None: continue
        h, w = frame.shape[:2]; x1,y1,x2,y2 = map(int, box)
        bw, bh = x2-x1, y2-y1
        if bw<=0 or bh<=0: continue
        px, py = int(bw*PAD), int(bh*PAD)
        x1=max(0,x1-px); y1=max(0,y1-py); x2=min(w,x2+px); y2=min(h,y2+py)
        crop = frame[y1:y2, x1:x2]
        if crop.size == 0: continue
        r = emo_model.predict(crop, imgsz=224, verbose=False)[0]
        p7 = r.probs.data.cpu().numpy()
        p4 = p7[FER_IDX4]; s = p4.sum()
        if s > 0: acc += p4/s; used += 1
    cap.release()
    return (acc/used) if used > 0 else None

def plot_confusion(cm, labels, title, fname):
    cmn = cm.astype(float)/ (cm.sum(axis=1, keepdims=True)+1e-9) * 100
    fig, ax = plt.subplots(figsize=(4.2,3.6))
    im = ax.imshow(cmn, cmap="Blues", vmin=0, vmax=100)
    ax.set_xticks(range(len(labels))); ax.set_xticklabels(labels, rotation=45, ha="right")
    ax.set_yticks(range(len(labels))); ax.set_yticklabels(labels)
    ax.set_xlabel("Predicted"); ax.set_ylabel("Ground Truth"); ax.set_title(title)
    for i in range(len(labels)):
        for j in range(len(labels)):
            ax.text(j, i, f"{cmn[i,j]:.0f}", ha="center", va="center",
                    color="white" if cmn[i,j]>50 else "black", fontsize=8)
    fig.colorbar(im, fraction=0.046, pad=0.04); fig.tight_layout()
    fig.savefig(fname, dpi=200); plt.show()
    print("saved", fname)

print("\u2705 helpers ready")


## PART A - Speech Emotion Recognition metrics on RAVDESS audio
Closes point A (SER evaluated on a standard audio dataset).

In [ ]:
wavs = glob.glob(os.path.join(RAVDESS_AUDIO_DIR, "**", "*.wav"), recursive=True)
assert wavs, f"No .wav found under {RAVDESS_AUDIO_DIR} - check the path."
print(f"found {len(wavs)} audio files")

per_class = defaultdict(int)
y_true, y_pred = [], []
for i, p in enumerate(sorted(wavs)):
    emo, _ = parse_emotion(p)
    if emo not in RAV_EMO: continue
    gt = RAV_EMO[emo]
    if AUDIO_MAX_PER_CLASS and per_class[gt] >= AUDIO_MAX_PER_CLASS: continue
    per_class[gt] += 1
    pr = speech_probs4(p)
    y_true.append(gt); y_pred.append(COMMON4[int(np.argmax(pr))])
    if (i+1) % 100 == 0: print(f"  {i+1}/{len(wavs)} processed")

acc = accuracy_score(y_true, y_pred)
mf1 = f1_score(y_true, y_pred, average="macro", labels=COMMON4)
wf1 = f1_score(y_true, y_pred, average="weighted", labels=COMMON4)
print("\n================ SER RESULTS (paper Table) ================")
print(f"clips evaluated : {len(y_true)}  | per-class: {dict(per_class)}")
print(f"Accuracy        : {acc*100:.1f}%")
print(f"Macro-F1        : {mf1:.3f}")
print(f"Weighted-F1     : {wf1:.3f}")
print("\n" + classification_report(y_true, y_pred, labels=COMMON4, digits=3))
cm = confusion_matrix(y_true, y_pred, labels=COMMON4)
plot_confusion(cm, COMMON4, "RAVDESS SER (HuBERT) confusion %", os.path.join(OUT_DIR, "ser_confusion.png"))
with open(os.path.join(OUT_DIR, "ser_metrics.csv"), "w", newline="") as f:
    w = csv.writer(f); w.writerow(["metric","value"])
    w.writerow(["accuracy", f"{acc*100:.2f}"]); w.writerow(["macro_f1", f"{mf1:.4f}"]); w.writerow(["weighted_f1", f"{wf1:.4f}"]); w.writerow(["n_clips", len(y_true)])
print("saved ser_metrics.csv")


## PART B - Multimodal late-fusion ablation
Closes point B. Pairs each spoken utterance's audio with its video, builds vision and speech probability vectors, then sweeps alpha. Also tests the confidence-weighted dynamic-alpha rule from Section III.H.

In [ ]:
have_video = os.path.isdir(RAVDESS_VIDEO_DIR) and bool(glob.glob(os.path.join(RAVDESS_VIDEO_DIR, "**", "*.mp4"), recursive=True))
if not have_video:
    print("Video set not found at", RAVDESS_VIDEO_DIR)
    print("Download Video_Speech_Actors_01-24.zip and set RAVDESS_VIDEO_DIR to run Part B.")
else:
    vids = glob.glob(os.path.join(RAVDESS_VIDEO_DIR, "**", "*.mp4"), recursive=True)
    vid_by_key = {}
    for v in vids:
        emo, key = parse_emotion(v)
        if emo in RAV_EMO: vid_by_key[key] = v
    # build matched audio+video list, capped per class
    pairs, cnt = [], defaultdict(int)
    for a in sorted(glob.glob(os.path.join(RAVDESS_AUDIO_DIR, "**", "*.wav"), recursive=True)):
        emo, key = parse_emotion(a)
        if emo not in RAV_EMO or key not in vid_by_key: continue
        gt = RAV_EMO[emo]
        if cnt[gt] >= FUSION_MAX_PER_CLASS: continue
        cnt[gt] += 1; pairs.append((a, vid_by_key[key], gt))
    print(f"matched audio+video pairs: {len(pairs)} | per-class: {dict(cnt)}")

    face_list, speech_list, gts = [], [], []
    for i, (a, v, gt) in enumerate(pairs):
        fp = face_probs4_from_video(v)
        if fp is None: continue
        sp = speech_probs4(a)
        face_list.append(fp); speech_list.append(sp); gts.append(COMMON4.index(gt))
        if (i+1) % 50 == 0: print(f"  {i+1}/{len(pairs)} pairs done")
    F = np.array(face_list); S = np.array(speech_list); Y = np.array(gts)
    print(f"usable pairs (face detected): {len(Y)}")

    def acc_of(P): return float((P.argmax(1) == Y).mean())
    alphas = np.round(np.arange(0.0, 1.0001, 0.1), 2)
    rows = []
    for al in alphas:
        rows.append((float(al), acc_of(al*F + (1-al)*S)))
    speech_only = acc_of(S); vision_only = acc_of(F)
    # confidence-weighted dynamic alpha (Section III.H): a = conf_face / (conf_face + conf_speech)
    fc = F.max(1, keepdims=True); sc = S.max(1, keepdims=True)
    a_dyn = fc / (fc + sc + 1e-9)
    dyn_acc = acc_of(a_dyn*F + (1-a_dyn)*S)
    best_al, best_acc = max(rows, key=lambda r: r[1])

    print("\n============ FUSION ABLATION (paper Table) ============")
    print(f"{"alpha":>6} {"fused_acc":>10}")
    for al, ac in rows: print(f"{al:>6.1f} {ac*100:>9.1f}%")
    print("-"*22)
    print(f"speech-only (a=0.0): {speech_only*100:.1f}%")
    print(f"vision-only (a=1.0): {vision_only*100:.1f}%")
    print(f"best fixed alpha   : a={best_al:.1f} -> {best_acc*100:.1f}%")
    print(f"dynamic conf-weight: {dyn_acc*100:.1f}%")

    with open(os.path.join(OUT_DIR, "fusion_ablation.csv"), "w", newline="") as f:
        w = csv.writer(f); w.writerow(["alpha","fused_accuracy_pct"])
        for al, ac in rows: w.writerow([f"{al:.1f}", f"{ac*100:.2f}"])
        w.writerow(["dynamic", f"{dyn_acc*100:.2f}"])
    fig, ax = plt.subplots(figsize=(5,3.4))
    ax.plot([r[0] for r in rows], [r[1]*100 for r in rows], "-o", label="fixed alpha")
    ax.axhline(dyn_acc*100, ls="--", color="green", label=f"dynamic ({dyn_acc*100:.1f}%)")
    ax.set_xlabel("alpha (vision weight)"); ax.set_ylabel("accuracy %")
    ax.set_title("Late-fusion ablation on RAVDESS (4-class)"); ax.legend(); ax.grid(alpha=0.3)
    fig.tight_layout(); fig.savefig(os.path.join(OUT_DIR, "fusion_ablation.png"), dpi=200); plt.show()
    print("saved fusion_ablation.png and fusion_ablation.csv")


## What to send me back
Once these run, paste the two result blocks (the SER results and the fusion ablation table). Those numbers become the new Table for the speech branch and the fusion-ablation table in the paper, and the two PNGs in `eval_outputs/` become the new figures. Then I do the paper edits: the multimodal results section, the fusion table, plus the pure-text fixes (Table VI disclaimer, FER-2013 limitation paragraph, demographic/intersectional ethics note, K=30 in Table IV, SER/TTA acronym definitions, and reference formatting).